In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [35]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent


In [11]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()

In [12]:
len(docs)

9

In [13]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_docs = splitter.split_documents(docs)

In [14]:
len(splitted_docs)

26

In [15]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = InMemoryVectorStore.from_documents(
    documents=splitted_docs,
    embedding=embeddings
)

In [19]:
### agent = tools, llm, prompt

In [70]:

@tool
def retriever_tool(query:str):
    """
        This tool can help you to retrieve the relevant data of the PDF Documents, and these pdf
        documents have details about medical reports.
    """

    print("Tool Called: ", query)
    docs = vector_store.similarity_search(query=query, k = 4)
    context = ""

    for doc in docs:
        context = doc.page_content + "\n\n"
    
    return context




In [55]:
llm = ChatOpenAI(model="gpt-5")

In [61]:
System_Prompt = """
    You are a helpful assistant that answers questions using retrieved context.
	ALWAYS use the `retriever_tool` tool for questions requiring external knowledge.
"""

In [71]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=System_Prompt
)

In [72]:
query = "What is the name of patient, and what is the name of Doctors"
response = agent.invoke({"messages":[{"role":"user", "content":query}]})

Tool Called:  patient name and doctor name
Tool Called:  Doctor name
Tool Called:  Patient Name


In [73]:
result = response["messages"][-1].content

In [74]:
print(result)

- Patient name: Ms. Nikita Chudhary
- Doctors: 
  - Dr. Nitin Nahar (Referring Physician)
  - Dr. Sunanda, MD (Sr. Consultant Pathologist, Hematology & Immunology)
